In [1]:
import tubesml as tml
import pandas as pd
import numpy as np

from source.report import _point_to_proba

from sklearn.metrics import brier_score_loss, mean_squared_error

from sklearn.model_selection import KFold

from sklearn.linear_model import Ridge, LogisticRegression, Lasso
from sklearn.pipeline import Pipeline

import lightgbm as lgb
import xgboost as xgb

import optuna
from optuna.samplers import TPESampler

In [2]:
df = pd.read_csv('data/processed/men_training.csv')

N_FOLDS = 5
kfolds = KFold(n_splits=N_FOLDS, shuffle=True, random_state=13)

df_train, df_test = tml.make_test(df, test_size=0.2, random_state=34)

DROP = ["target", "target_points", "ID", "DayNum", "Team1", "Team2",
        'T1_Loc', 'T2_Loc',
                "T1_region", "T2_region", "Season", "delta_Loc",
                "Season", "competitive", "competitive_score",
                "delta_def_rating_diff", "delta_impact_diff",
                "T1_def_rating_diff", "T2_def_rating_diff"]

df_train.head()

df_train = df.copy()

## Feats cats

In [3]:
all_feats = [c for c in df_train if c not in DROP]
all_feats

['T1_Ast',
 'T1_Ast_TO_ratio',
 'T1_Ast_TO_ratio_diff',
 'T1_Ast_diff',
 'T1_Away',
 'T1_Blk',
 'T1_Blk_diff',
 'T1_DR',
 'T1_DR_diff',
 'T1_DR_opportunity',
 'T1_DR_opportunity_diff',
 'T1_Eff_FG_perc_diff',
 'T1_FG3_ratio',
 'T1_FG3_ratio_diff',
 'T1_FGA',
 'T1_FGA2',
 'T1_FGA2_diff',
 'T1_FGA3',
 'T1_FGA3_diff',
 'T1_FGA_diff',
 'T1_FGM',
 'T1_FGM2',
 'T1_FGM2_diff',
 'T1_FGM3',
 'T1_FGM3_diff',
 'T1_FGM_diff',
 'T1_FGM_no_ast',
 'T1_FGM_no_ast_diff',
 'T1_FTA',
 'T1_FTA_diff',
 'T1_FTM',
 'T1_FTM_diff',
 'T1_N_wins',
 'T1_OR',
 'T1_OR_diff',
 'T1_OR_opportunity',
 'T1_OR_opportunity_diff',
 'T1_OT_win',
 'T1_PF',
 'T1_PF_diff',
 'T1_Score',
 'T1_Score_diff',
 'T1_Stl',
 'T1_Stl_diff',
 'T1_TO',
 'T1_TO_diff',
 'T1_TO_perposs',
 'T1_TO_perposs_diff',
 'T1_Tot_Reb',
 'T1_Tot_Reb_diff',
 'T1_True_shooting_perc_diff',
 'T1_def_rating',
 'T1_impact',
 'T1_impact_diff',
 'T1_off_rating',
 'T1_off_rating_diff',
 'T1_opp_FGA',
 'T1_opp_FGM',
 'T1_opp_FGM3',
 'T1_opp_FTA',
 'T1_opp_PF',
 'T

In [4]:
all_delta = [c for c in df_train if c not in DROP and "delta" in c]

all_delta

['delta_Ast',
 'delta_Ast_TO_ratio',
 'delta_Ast_TO_ratio_diff',
 'delta_Ast_diff',
 'delta_Away',
 'delta_Blk',
 'delta_Blk_diff',
 'delta_DR',
 'delta_DR_diff',
 'delta_DR_opportunity',
 'delta_DR_opportunity_diff',
 'delta_Eff_FG_perc_diff',
 'delta_FG3_ratio',
 'delta_FG3_ratio_diff',
 'delta_FGA',
 'delta_FGA2',
 'delta_FGA2_diff',
 'delta_FGA3',
 'delta_FGA3_diff',
 'delta_FGA_diff',
 'delta_FGM',
 'delta_FGM2',
 'delta_FGM2_diff',
 'delta_FGM3',
 'delta_FGM3_diff',
 'delta_FGM_diff',
 'delta_FGM_no_ast',
 'delta_FGM_no_ast_diff',
 'delta_FTA',
 'delta_FTA_diff',
 'delta_FTM',
 'delta_FTM_diff',
 'delta_N_wins',
 'delta_OR',
 'delta_OR_diff',
 'delta_OR_opportunity',
 'delta_OR_opportunity_diff',
 'delta_OT_win',
 'delta_PF',
 'delta_PF_diff',
 'delta_Score',
 'delta_Score_diff',
 'delta_Stl',
 'delta_Stl_diff',
 'delta_TO',
 'delta_TO_diff',
 'delta_TO_perposs',
 'delta_TO_perposs_diff',
 'delta_Tot_Reb',
 'delta_Tot_Reb_diff',
 'delta_True_shooting_perc_diff',
 'delta_def_ratin

In [5]:
no_delta = [c for c in df_train if c not in DROP and "delta" not in c]
no_delta

['T1_Ast',
 'T1_Ast_TO_ratio',
 'T1_Ast_TO_ratio_diff',
 'T1_Ast_diff',
 'T1_Away',
 'T1_Blk',
 'T1_Blk_diff',
 'T1_DR',
 'T1_DR_diff',
 'T1_DR_opportunity',
 'T1_DR_opportunity_diff',
 'T1_Eff_FG_perc_diff',
 'T1_FG3_ratio',
 'T1_FG3_ratio_diff',
 'T1_FGA',
 'T1_FGA2',
 'T1_FGA2_diff',
 'T1_FGA3',
 'T1_FGA3_diff',
 'T1_FGA_diff',
 'T1_FGM',
 'T1_FGM2',
 'T1_FGM2_diff',
 'T1_FGM3',
 'T1_FGM3_diff',
 'T1_FGM_diff',
 'T1_FGM_no_ast',
 'T1_FGM_no_ast_diff',
 'T1_FTA',
 'T1_FTA_diff',
 'T1_FTM',
 'T1_FTM_diff',
 'T1_N_wins',
 'T1_OR',
 'T1_OR_diff',
 'T1_OR_opportunity',
 'T1_OR_opportunity_diff',
 'T1_OT_win',
 'T1_PF',
 'T1_PF_diff',
 'T1_Score',
 'T1_Score_diff',
 'T1_Stl',
 'T1_Stl_diff',
 'T1_TO',
 'T1_TO_diff',
 'T1_TO_perposs',
 'T1_TO_perposs_diff',
 'T1_Tot_Reb',
 'T1_Tot_Reb_diff',
 'T1_True_shooting_perc_diff',
 'T1_def_rating',
 'T1_impact',
 'T1_impact_diff',
 'T1_off_rating',
 'T1_off_rating_diff',
 'T1_opp_FGA',
 'T1_opp_FGM',
 'T1_opp_FGM3',
 'T1_opp_FTA',
 'T1_opp_PF',
 'T

In [6]:
seeds = [c for c in df_train if c not in DROP and "Seed" in c] + [c for c in df_train if "quality" in c] + [c for c in df_train if "stage" in c] + [c for c in df_train if "elo" in c]
seeds

['T1_Seed',
 'T2_Seed',
 'delta_Seed',
 'T1_quality',
 'T2_quality',
 'delta_quality',
 'stage_Round1',
 'stage_Round2',
 'stage_Round3',
 'stage_Round4',
 'stage_final',
 'stage_finalfour',
 'stage_impossible',
 'T1_elo',
 'T2_elo',
 'delta_elo']

In [7]:
no_seeds = [c for c in df_train if c not in DROP and "Seed" not in c]
no_seeds

['T1_Ast',
 'T1_Ast_TO_ratio',
 'T1_Ast_TO_ratio_diff',
 'T1_Ast_diff',
 'T1_Away',
 'T1_Blk',
 'T1_Blk_diff',
 'T1_DR',
 'T1_DR_diff',
 'T1_DR_opportunity',
 'T1_DR_opportunity_diff',
 'T1_Eff_FG_perc_diff',
 'T1_FG3_ratio',
 'T1_FG3_ratio_diff',
 'T1_FGA',
 'T1_FGA2',
 'T1_FGA2_diff',
 'T1_FGA3',
 'T1_FGA3_diff',
 'T1_FGA_diff',
 'T1_FGM',
 'T1_FGM2',
 'T1_FGM2_diff',
 'T1_FGM3',
 'T1_FGM3_diff',
 'T1_FGM_diff',
 'T1_FGM_no_ast',
 'T1_FGM_no_ast_diff',
 'T1_FTA',
 'T1_FTA_diff',
 'T1_FTM',
 'T1_FTM_diff',
 'T1_N_wins',
 'T1_OR',
 'T1_OR_diff',
 'T1_OR_opportunity',
 'T1_OR_opportunity_diff',
 'T1_OT_win',
 'T1_PF',
 'T1_PF_diff',
 'T1_Score',
 'T1_Score_diff',
 'T1_Stl',
 'T1_Stl_diff',
 'T1_TO',
 'T1_TO_diff',
 'T1_TO_perposs',
 'T1_TO_perposs_diff',
 'T1_Tot_Reb',
 'T1_Tot_Reb_diff',
 'T1_True_shooting_perc_diff',
 'T1_def_rating',
 'T1_impact',
 'T1_impact_diff',
 'T1_off_rating',
 'T1_off_rating_diff',
 'T1_opp_FGA',
 'T1_opp_FGM',
 'T1_opp_FGM3',
 'T1_opp_FTA',
 'T1_opp_PF',
 'T

In [8]:
feats_dict = {"all_feats": all_feats,
              "all_delta": all_delta, "no_delta": no_delta, "no_seeds": no_seeds, "seeds": seeds}

# Points predictions

## LGBM

In [9]:
def objective(trial, data=df_train, target=df_train["target_points"]):
    param = {
        "max_depth": trial.suggest_int("max_depth", 3, 200),
        "num_leaves": trial.suggest_int("num_leaves", 10, 100),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 100.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 100.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1),
        'subsample': trial.suggest_float('subsample', 0.4, 1),
        'min_child_weight': trial.suggest_float('min_child_weight', 1e-3 , 300),
        'feats': trial.suggest_categorical("feats", list(feats_dict.keys())),
        "clip_val": trial.suggest_int("clip_val", 20, 50),
        "padd": trial.suggest_float("padd", 0, 0.05)
        # "agg_func": trial.suggest_categorical("agg_func", ["mean", "median", "std"]),
        # "formula": trial.suggest_categorical("formula", [True, False])
    }
    
        
    train = data.copy()

    FEATURES = feats_dict[param["feats"]]


    num_pipe = Pipeline([('sel', tml.DtypeSel('numeric')),
                        ('imputer', tml.DfImputer(strategy='mean')),
                        ])

    processing_pipe = tml.FeatureUnionDf(transformer_list=[('num', num_pipe),
                                                        #('cat_means', cat_pipe)
                                                        ])
        
    model = lgb.LGBMRegressor(random_state=34, n_jobs=-1, verbose=-1, n_estimators=10000,
                              learning_rate=0.1,
                             colsample_bytree=param["colsample_bytree"],
                             min_child_weight=param['min_child_weight'],
                             reg_lambda=param['reg_lambda'],
                             reg_alpha=param['reg_alpha'],
                             subsample=param['subsample'],
                             num_leaves=param["num_leaves"],
                             max_depth=param['max_depth'],
                             eval_metric="l2")

    pipe = Pipeline([ #("fe", FeatEng(formula=param["formula"])),
                        ("processing", processing_pipe),

                    ("model", model)])
    
    callbacks = [lgb.early_stopping(100, verbose=0)]
    
    fit_params = {"callbacks":callbacks, "eval_metric": "l2"}

    cvscore = tml.CrossValidate(data=train[FEATURES], target=target, cv=kfolds, estimator=pipe, fit_params=fit_params, early_stopping=True)
    oof, _ = cvscore.score()

    spline_oof, _ = _point_to_proba(oof, target, oof, clip_val=param["clip_val"], padd=param["padd"])

    score = brier_score_loss(np.where(target > 0, 1, 0), y_prob=spline_oof)
    
    return score

In [10]:
sampler = TPESampler(seed=645)  # Make the sampler behave in a deterministic way.

study = optuna.create_study(direction='minimize', sampler=sampler)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=1000, n_jobs=-1)
print('Number of finished trials:', len(study.trials))
print('Best trial:', study.best_trial.params)

[I 2026-03-06 22:44:16,079] A new study created in memory with name: no-name-c4496105-8499-4fdd-8fc6-7c735da3b46a


Number of finished trials: 1000
Best trial: {'max_depth': 115, 'num_leaves': 97, 'reg_lambda': 98.18828303496446, 'reg_alpha': 64.62473005598024, 'colsample_bytree': 0.33539785257661636, 'subsample': 0.8098921917347464, 'min_child_weight': 79.70139276027618, 'feats': 'all_feats', 'clip_val': 22, 'padd': 0.03590376956201806}


In [11]:
0.183152

0.183152

In [12]:
study.trials_dataframe().sort_values('value', ascending=True).head(20)

,number,value,datetime_start,datetime_complete,duration,params_clip_val,params_colsample_bytree,params_feats,params_max_depth,params_min_child_weight,params_num_leaves,params_padd,params_reg_alpha,params_reg_lambda,params_subsample,state
624,624,0.182903,2026-03-06 23:14:21.266198,2026-03-06 23:15:18.410385,0 days 00:00:57.144187,22,0.335398,all_feats,115,79.701393,97,0.035904,64.624730,98.188283,0.809892,COMPLETE
741,741,0.182908,2026-03-06 23:20:24.168254,2026-03-06 23:21:22.159271,0 days 00:00:57.991017,22,0.373942,all_feats,102,229.289338,48,0.042358,98.307051,96.466202,0.690531,COMPLETE
403,403,0.182972,2026-03-06 23:02:10.671458,2026-03-06 23:03:01.600993,0 days 00:00:50.929535,24,0.405336,all_feats,47,241.878309,99,0.017081,75.454278,89.697202,0.932746,COMPLETE
956,956,0.183039,2026-03-06 23:32:02.803434,2026-03-06 23:33:10.305785,0 days 00:01:07.502351,22,0.353107,all_feats,162,237.140614,88,0.038600,87.646399,99.885727,0.624815,COMPLETE
103,103,0.183050,2026-03-06 22:47:32.734153,2026-03-06 22:48:15.043379,0 days 00:00:42.309226,24,0.360287,all_feats,95,266.210847,92,0.007318,56.372636,66.254717,0.706736,COMPLETE
914,914,0.183054,2026-03-06 23:29:39.211843,2026-03-06 23:30:40.812173,0 days 00:01:01.600330,21,0.370235,all_feats,100,237.290614,93,0.040765,94.977734,99.711592,0.586006,COMPLETE
719,719,0.183096,2026-03-06 23:19:10.329817,2026-03-06 23:20:04.072308,0 days 00:00:53.742491,22,0.364921,all_feats,94,244.290823,94,0.041522,96.715632,94.448710,0.698706,COMPLETE
909,909,0.183112,2026-03-06 23:29:20.919952,2026-03-06 23:30:22.106208,0 days 00:01:01.186256,21,0.389699,all_feats,89,237.028249,93,0.040952,66.345303,99.898761,0.688716,COMPLETE
913,913,0.183137,2026-03-06 23:29:34.906256,2026-03-06 23:30:41.159040,0 days 00:01:06.252784,21,0.391249,all_feats,100,239.059982,93,0.036570,94.903654,99.767847,0.686972,COMPLETE
776,776,0.183166,2026-03-06 23:22:27.506318,2026-03-06 23:23:23.483477,0 days 00:00:55.977159,23,0.338914,all_feats,106,243.413314,98,0.039178,99.893037,98.329993,0.678980,COMPLETE


## XGBoost

In [13]:
def objective(trial, data=df_train, target=df_train["target_points"]):
    param = {
        "max_depth": trial.suggest_int("max_depth", 3, 200),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 100.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 100.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1),
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.3, 1),
        'subsample': trial.suggest_float('subsample', 0.4, 1),
        'min_child_weight': trial.suggest_float('min_child_weight', 1e-3 , 300),
        'feats': trial.suggest_categorical("feats", list(feats_dict.keys())),
        "clip_val": trial.suggest_int("clip_val", 20, 50),
        "padd": trial.suggest_float("padd", 0, 0.05)
        # "agg_func": trial.suggest_categorical("agg_func", ["mean", "median", "std"]),
        # "formula": trial.suggest_categorical("formula", [True, False])
    }
    
        
    train = data.copy()

    FEATURES = feats_dict[param["feats"]]


    num_pipe = Pipeline([('sel', tml.DtypeSel('numeric')),
                        ('imputer', tml.DfImputer(strategy='mean')),
                        ])

    processing_pipe = tml.FeatureUnionDf(transformer_list=[('num', num_pipe),
                                                        #('cat_means', cat_pipe)
                                                        ])
        
    model = xgb.XGBRegressor(random_state=34, n_jobs=-1, n_estimators=10000,
                              learning_rate=0.1,
                             subsample=param["subsample"],
                            colsample_bytree=param["colsample_bytree"],
                            reg_alpha=param["reg_alpha"],
                            reg_lambda=param["reg_lambda"],
                            max_depth=param["max_depth"],
                            colsample_bylevel=param["colsample_bylevel"],
                            early_stopping_rounds=100,
                             eval_metric=mean_squared_error)

    pipe = Pipeline([ #("fe", FeatEng(formula=param["formula"])),
                        ("processing", processing_pipe),

                    ("model", model)])
    
    
    fit_params = {'verbose': False}

    cvscore = tml.CrossValidate(data=train[FEATURES], target=target, cv=kfolds, estimator=pipe, fit_params=fit_params, early_stopping=True)
    oof, _ = cvscore.score()

    spline_oof, _ = _point_to_proba(oof, target, oof, clip_val=param["clip_val"], padd=param["padd"])

    score = brier_score_loss(np.where(target > 0, 1, 0), y_prob=spline_oof)
    
    return score

In [14]:
sampler = TPESampler(seed=645)  # Make the sampler behave in a deterministic way.

study = optuna.create_study(direction='minimize', sampler=sampler)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=1000, n_jobs=-1)
print('Number of finished trials:', len(study.trials))
print('Best trial:', study.best_trial.params)

study.trials_dataframe().sort_values('value', ascending=True).head(20)

Number of finished trials: 1000
Best trial: {'max_depth': 3, 'reg_lambda': 39.789794421560345, 'reg_alpha': 85.8182097912694, 'colsample_bytree': 0.5630686409349378, 'colsample_bylevel': 0.718398210343427, 'subsample': 0.7659607202394906, 'min_child_weight': 53.36080845841707, 'feats': 'all_feats', 'clip_val': 46, 'padd': 0.013121156753474193}


,number,value,datetime_start,datetime_complete,duration,params_clip_val,params_colsample_bylevel,params_colsample_bytree,params_feats,params_max_depth,params_min_child_weight,params_padd,params_reg_alpha,params_reg_lambda,params_subsample,state
255,255,0.182957,2026-03-07 00:46:24.552986,2026-03-07 00:48:24.599135,0 days 00:02:00.046149,46,0.718398,0.563069,all_feats,3,53.360808,0.013121,85.818210,39.789794,0.765961,COMPLETE
922,922,0.183212,2026-03-07 03:05:33.339111,2026-03-07 03:07:39.090544,0 days 00:02:05.751433,39,0.736698,0.724283,all_feats,3,60.186918,0.033023,29.111970,41.851253,0.738350,COMPLETE
852,852,0.183307,2026-03-07 02:52:48.077972,2026-03-07 02:55:13.544159,0 days 00:02:25.466187,50,0.743368,0.608315,all_feats,3,67.355152,0.035122,29.592509,37.557918,0.750010,COMPLETE
182,182,0.183308,2026-03-07 00:32:35.461454,2026-03-07 00:34:58.644848,0 days 00:02:23.183394,41,0.637992,0.657917,all_feats,3,2.334987,0.037813,35.701048,42.624839,0.906603,COMPLETE
383,383,0.183397,2026-03-07 01:10:27.376660,2026-03-07 01:12:46.127538,0 days 00:02:18.750878,37,0.715444,0.635529,all_feats,3,146.647496,0.044816,85.435701,51.281779,0.899817,COMPLETE
690,690,0.183413,2026-03-07 02:16:16.359908,2026-03-07 02:18:43.173310,0 days 00:02:26.813402,41,0.678094,0.572924,all_feats,3,209.084450,0.038720,56.140031,44.529159,0.776163,COMPLETE
187,187,0.183474,2026-03-07 00:33:37.802233,2026-03-07 00:35:47.509946,0 days 00:02:09.707713,41,0.989868,0.613485,all_feats,3,37.357302,0.037259,55.921715,47.013692,0.754855,COMPLETE
989,989,0.183478,2026-03-07 03:17:48.351388,2026-03-07 03:20:05.452641,0 days 00:02:17.101253,38,0.694812,0.613912,all_feats,3,60.349319,0.034710,12.743465,37.339502,0.735346,COMPLETE
507,507,0.183568,2026-03-07 01:37:19.512833,2026-03-07 01:39:33.238153,0 days 00:02:13.725320,41,0.667684,0.571396,all_feats,3,15.941429,0.036433,13.365146,51.912506,0.831068,COMPLETE
623,623,0.183601,2026-03-07 02:01:11.441406,2026-03-07 02:03:26.175168,0 days 00:02:14.733762,42,0.685652,0.477109,all_feats,3,54.782465,0.039348,34.047073,35.804257,0.759956,COMPLETE


## Ridge

In [11]:
def objective(trial, data=df_train, target=df_train["target_points"]):
    param = {
        "alpha": trial.suggest_float("alpha", 0.1, 200),
        'feats': trial.suggest_categorical("feats", list(feats_dict.keys())),
        "clip_val": trial.suggest_int("clip_val", 20, 50),
        "padd": trial.suggest_float("padd", 0, 0.05)
    }
    
        
    train = data.copy()

    FEATURES = feats_dict[param["feats"]]


    num_pipe = Pipeline([('sel', tml.DtypeSel('numeric')),
                        ('imputer', tml.DfImputer(strategy='mean')),
                        ])

    processing_pipe = tml.FeatureUnionDf(transformer_list=[('num', num_pipe),
                                                        #('cat_means', cat_pipe)
                                                        ])
        
    model = Ridge(random_state=34, alpha=param["alpha"], max_iter=10000)

    pipe = Pipeline([ #("fe", FeatEng(formula=param["formula"])),
                        ("processing", processing_pipe),
                        ("scaler", tml.DfScaler()),

                    ("model", model)])
    

    cvscore = tml.CrossValidate(data=train[FEATURES], target=target, cv=kfolds, estimator=pipe)
    oof, _ = cvscore.score()

    spline_oof, _ = _point_to_proba(oof, target, oof, clip_val=param["clip_val"], padd=param["padd"])

    score = brier_score_loss(np.where(target > 0, 1, 0), y_prob=spline_oof)
    
    return score

In [12]:
sampler = TPESampler(seed=645)  # Make the sampler behave in a deterministic way.

study = optuna.create_study(direction='minimize', sampler=sampler)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=2000, n_jobs=-1)
print('Number of finished trials:', len(study.trials))
print('Best trial:', study.best_trial.params)

study.trials_dataframe().sort_values('value', ascending=True).head(20)

Number of finished trials: 2000
Best trial: {'alpha': 0.10151685999354834, 'feats': 'all_delta', 'clip_val': 21, 'padd': 0.025912677098583754}


,number,value,datetime_start,datetime_complete,duration,params_alpha,params_clip_val,params_feats,params_padd,state
1804,1804,0.181978,2026-03-07 12:40:45.927633,2026-03-07 12:40:57.528197,0 days 00:00:11.600564,0.101517,21,all_delta,0.025913,COMPLETE
1760,1760,0.181979,2026-03-07 12:40:03.867165,2026-03-07 12:40:16.248465,0 days 00:00:12.381300,0.102785,20,all_delta,0.022161,COMPLETE
760,760,0.181979,2026-03-07 12:25:30.421447,2026-03-07 12:25:42.043181,0 days 00:00:11.621734,0.103011,20,all_delta,0.020601,COMPLETE
1600,1600,0.181979,2026-03-07 12:37:28.107592,2026-03-07 12:37:39.526974,0 days 00:00:11.419382,0.100816,21,all_delta,0.016192,COMPLETE
631,631,0.181979,2026-03-07 12:23:13.847352,2026-03-07 12:23:27.918025,0 days 00:00:14.070673,0.101668,21,all_delta,0.029000,COMPLETE
935,935,0.181979,2026-03-07 12:27:43.988142,2026-03-07 12:27:57.023337,0 days 00:00:13.035195,0.103622,20,all_delta,0.022627,COMPLETE
1418,1418,0.181979,2026-03-07 12:35:06.382753,2026-03-07 12:35:18.542720,0 days 00:00:12.159967,0.101191,21,all_delta,0.015926,COMPLETE
1544,1544,0.181980,2026-03-07 12:36:51.346823,2026-03-07 12:37:03.662572,0 days 00:00:12.315749,0.103509,20,all_delta,0.017471,COMPLETE
773,773,0.181980,2026-03-07 12:25:40.011536,2026-03-07 12:25:52.041336,0 days 00:00:12.029800,0.105301,21,all_delta,0.023673,COMPLETE
1797,1797,0.181981,2026-03-07 12:40:40.282242,2026-03-07 12:40:53.003923,0 days 00:00:12.721681,0.106144,21,all_delta,0.025907,COMPLETE


## Lasso

In [13]:
def objective(trial, data=df_train, target=df_train["target_points"]):
    param = {
        "alpha": trial.suggest_float("alpha", 0.1, 200),
        'feats': trial.suggest_categorical("feats", list(feats_dict.keys())),
        "clip_val": trial.suggest_int("clip_val", 20, 50),
        "padd": trial.suggest_float("padd", 0, 0.05)
    }
    
        
    train = data.copy()

    FEATURES = feats_dict[param["feats"]]


    num_pipe = Pipeline([('sel', tml.DtypeSel('numeric')),
                        ('imputer', tml.DfImputer(strategy='mean')),
                        ])

    processing_pipe = tml.FeatureUnionDf(transformer_list=[('num', num_pipe),
                                                        #('cat_means', cat_pipe)
                                                        ])
        
    model = Lasso(random_state=34, alpha=param["alpha"], max_iter=10000)

    pipe = Pipeline([ #("fe", FeatEng(formula=param["formula"])),
                        ("processing", processing_pipe),
                        ("scaler", tml.DfScaler()),

                    ("model", model)])
    

    cvscore = tml.CrossValidate(data=train[FEATURES], target=target, cv=kfolds, estimator=pipe)
    oof, _ = cvscore.score()

    spline_oof, _ = _point_to_proba(oof, target, oof, clip_val=param["clip_val"], padd=param["padd"])

    score = brier_score_loss(np.where(target > 0, 1, 0), y_prob=spline_oof)
    
    return score

In [14]:
sampler = TPESampler(seed=645)  # Make the sampler behave in a deterministic way.

study = optuna.create_study(direction='minimize', sampler=sampler)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=2000, n_jobs=-1)
print('Number of finished trials:', len(study.trials))
print('Best trial:', study.best_trial.params)

study.trials_dataframe().sort_values('value', ascending=True).head(20)

is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
is_sparse is deprecated and will be removed in a future version. Check `

Number of finished trials: 2000
Best trial: {'alpha': 0.1011598148216751, 'feats': 'all_delta', 'clip_val': 32, 'padd': 0.023157492348947598}


,number,value,datetime_start,datetime_complete,duration,params_alpha,params_clip_val,params_feats,params_padd,state
1611,1611,0.183212,2026-03-07 13:12:34.873807,2026-03-07 13:12:49.251758,0 days 00:00:14.377951,0.101160,32,all_delta,0.023157,COMPLETE
1306,1306,0.183223,2026-03-07 13:07:33.471195,2026-03-07 13:07:45.529574,0 days 00:00:12.058379,0.100700,30,all_delta,0.000669,COMPLETE
523,523,0.183224,2026-03-07 12:55:22.015951,2026-03-07 12:55:33.990168,0 days 00:00:11.974217,0.100819,34,all_delta,0.001609,COMPLETE
830,830,0.183226,2026-03-07 13:00:04.955829,2026-03-07 13:00:18.370334,0 days 00:00:13.414505,0.101530,31,all_delta,0.008992,COMPLETE
1526,1526,0.183228,2026-03-07 13:10:53.821027,2026-03-07 13:11:08.015219,0 days 00:00:14.194192,0.108981,34,all_delta,0.028042,COMPLETE
1622,1622,0.183229,2026-03-07 13:12:41.216196,2026-03-07 13:12:55.883983,0 days 00:00:14.667787,0.110763,32,all_delta,0.030798,COMPLETE
699,699,0.183230,2026-03-07 12:57:56.010335,2026-03-07 12:58:12.236990,0 days 00:00:16.226655,0.104401,31,all_delta,0.014959,COMPLETE
505,505,0.183231,2026-03-07 12:55:06.041882,2026-03-07 12:55:18.869274,0 days 00:00:12.827392,0.103039,44,all_delta,0.003241,COMPLETE
1912,1912,0.183235,2026-03-07 13:17:30.255267,2026-03-07 13:17:43.579797,0 days 00:00:13.324530,0.104272,30,all_delta,0.009807,COMPLETE
1060,1060,0.183235,2026-03-07 13:03:38.977742,2026-03-07 13:03:53.287987,0 days 00:00:14.310245,0.105072,32,all_delta,0.013084,COMPLETE


# Probability Predictions


## LGBM

In [19]:
def objective(trial, data=df_train, target=df_train["target"]):
    param = {
        "max_depth": trial.suggest_int("max_depth", 3, 200),
        "num_leaves": trial.suggest_int("num_leaves", 10, 100),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 100.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 100.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1),
        'subsample': trial.suggest_float('subsample', 0.4, 1),
        'min_child_weight': trial.suggest_float('min_child_weight', 1e-3 , 300),
        'feats': trial.suggest_categorical("feats", list(feats_dict.keys())),
    }
    
        
    train = data.copy()

    FEATURES = feats_dict[param["feats"]]


    num_pipe = Pipeline([('sel', tml.DtypeSel('numeric')),
                        ('imputer', tml.DfImputer(strategy='mean')),
                        ])

    processing_pipe = tml.FeatureUnionDf(transformer_list=[('num', num_pipe),
                                                        #('cat_means', cat_pipe)
                                                        ])
        
    model = lgb.LGBMClassifier(random_state=34, n_jobs=-1, verbose=-1, n_estimators=10000,
                               learning_rate=0.1,
                             colsample_bytree=param["colsample_bytree"],
                             min_child_weight=param['min_child_weight'],
                             reg_lambda=param['reg_lambda'],
                             reg_alpha=param['reg_alpha'],
                             subsample=param['subsample'],
                             num_leaves=param["num_leaves"],
                             max_depth=param['max_depth'],
                             eval_metric="auc")

    pipe = Pipeline([ #("fe", FeatEng(formula=param["formula"])),
                        ("processing", processing_pipe),

                    ("model", model)])
    
    callbacks = [lgb.early_stopping(100, verbose=0)]
    
    fit_params = {"callbacks":callbacks, "eval_metric": "auc"}

    cvscore = tml.CrossValidate(data=train[FEATURES], target=target, cv=kfolds, estimator=pipe, fit_params=fit_params, early_stopping=True, predict_proba=True)
    oof, _ = cvscore.score()

    score = brier_score_loss(target, y_prob=oof)
    
    return score

In [20]:
sampler = TPESampler(seed=645)  # Make the sampler behave in a deterministic way.

study = optuna.create_study(direction='minimize', sampler=sampler)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=1000, n_jobs=-1)
print('Number of finished trials:', len(study.trials))
print('Best trial:', study.best_trial.params)

study.trials_dataframe().sort_values('value', ascending=True).head(20)

Number of finished trials: 1000
Best trial: {'max_depth': 92, 'num_leaves': 51, 'reg_lambda': 44.00536243237047, 'reg_alpha': 3.517393066057428, 'colsample_bytree': 0.4112546404929133, 'subsample': 0.898529310176569, 'min_child_weight': 47.85558942186549, 'feats': 'all_feats'}


,number,value,datetime_start,datetime_complete,duration,params_colsample_bytree,params_feats,params_max_depth,params_min_child_weight,params_num_leaves,params_reg_alpha,params_reg_lambda,params_subsample,state
897,897,0.186719,2026-03-07 04:36:45.036229,2026-03-07 04:37:23.174325,0 days 00:00:38.138096,0.411255,all_feats,92,47.855589,51,3.517393,44.005362,0.898529,COMPLETE
884,884,0.187136,2026-03-07 04:36:16.578492,2026-03-07 04:36:47.198538,0 days 00:00:30.620046,0.407672,all_feats,94,51.406472,46,1.748187,0.867684,0.897356,COMPLETE
887,887,0.187146,2026-03-07 04:36:24.714915,2026-03-07 04:36:58.384573,0 days 00:00:33.669658,0.345501,all_feats,93,53.048506,47,0.033628,47.194002,0.973086,COMPLETE
653,653,0.187184,2026-03-07 04:28:38.906368,2026-03-07 04:29:10.299023,0 days 00:00:31.392655,0.314533,all_feats,88,47.624331,51,1.685361,46.770161,0.932810,COMPLETE
734,734,0.187189,2026-03-07 04:31:29.729335,2026-03-07 04:32:02.983681,0 days 00:00:33.254346,0.347716,all_feats,88,44.749334,51,1.523920,52.038074,0.897639,COMPLETE
516,516,0.187248,2026-03-07 04:24:23.691570,2026-03-07 04:24:53.385911,0 days 00:00:29.694341,0.350284,all_feats,98,44.902064,78,3.232698,49.731937,0.889971,COMPLETE
479,479,0.187258,2026-03-07 04:23:21.947479,2026-03-07 04:23:47.433756,0 days 00:00:25.486277,0.371985,all_feats,160,33.903735,88,0.575452,46.353491,0.964723,COMPLETE
635,635,0.187271,2026-03-07 04:28:01.396043,2026-03-07 04:28:34.786178,0 days 00:00:33.390135,0.317591,all_feats,97,30.360887,45,0.020070,46.551021,0.910511,COMPLETE
645,645,0.187303,2026-03-07 04:28:22.074692,2026-03-07 04:28:54.654826,0 days 00:00:32.580134,0.386529,all_feats,66,46.834857,49,1.479965,47.422417,0.865707,COMPLETE
668,668,0.187320,2026-03-07 04:29:09.552888,2026-03-07 04:29:46.447713,0 days 00:00:36.894825,0.366288,all_feats,50,48.592149,50,1.726721,55.643695,0.859652,COMPLETE


## XGBoost

In [21]:
def objective(trial, data=df_train, target=df_train["target"]):
    param = {
        "max_depth": trial.suggest_int("max_depth", 3, 200),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 100.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 100.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1),
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.3, 1),
        'subsample': trial.suggest_float('subsample', 0.4, 1),
        'min_child_weight': trial.suggest_float('min_child_weight', 1e-3 , 300),
        'feats': trial.suggest_categorical("feats", list(feats_dict.keys())),
    }
    
        
    train = data.copy()

    FEATURES = feats_dict[param["feats"]]


    num_pipe = Pipeline([('sel', tml.DtypeSel('numeric')),
                        ('imputer', tml.DfImputer(strategy='mean')),
                        ])

    processing_pipe = tml.FeatureUnionDf(transformer_list=[('num', num_pipe),
                                                        #('cat_means', cat_pipe)
                                                        ])
        
    model = xgb.XGBClassifier(random_state=34, n_jobs=-1, n_estimators=10000,
                              learning_rate=0.1,
                             subsample=param["subsample"],
                            colsample_bytree=param["colsample_bytree"],
                            reg_alpha=param["reg_alpha"],
                            reg_lambda=param["reg_lambda"],
                            max_depth=param["max_depth"],
                            colsample_bylevel=param["colsample_bylevel"],
                            early_stopping_rounds=100,
                             eval_metric="auc")

    pipe = Pipeline([ #("fe", FeatEng(formula=param["formula"])),
                        ("processing", processing_pipe),

                    ("model", model)])
    
    
    fit_params = {"verbose": False}

    cvscore = tml.CrossValidate(data=train[FEATURES], target=target, cv=kfolds, estimator=pipe, fit_params=fit_params, early_stopping=True, predict_proba=True)
    oof, _ = cvscore.score()

    score = brier_score_loss(target, y_prob=oof)
    
    return score

In [22]:
sampler = TPESampler(seed=645)  # Make the sampler behave in a deterministic way.

study = optuna.create_study(direction='minimize', sampler=sampler)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=1000, n_jobs=-1)
print('Number of finished trials:', len(study.trials))
print('Best trial:', study.best_trial.params)

study.trials_dataframe().sort_values('value', ascending=True).head(20)

Number of finished trials: 1000
Best trial: {'max_depth': 108, 'reg_lambda': 30.935315773225398, 'reg_alpha': 8.524310782577258, 'colsample_bytree': 0.4914753840095336, 'colsample_bylevel': 0.3002201650698665, 'subsample': 0.400991101026403, 'min_child_weight': 131.93444263029593, 'feats': 'seeds'}


,number,value,datetime_start,datetime_complete,duration,params_colsample_bylevel,params_colsample_bytree,params_feats,params_max_depth,params_min_child_weight,params_reg_alpha,params_reg_lambda,params_subsample,state
914,914,0.188007,2026-03-07 05:22:45.850325,2026-03-07 05:23:13.491791,0 days 00:00:27.641466,0.300220,0.491475,seeds,108,131.934443,8.524311,30.935316,0.400991,COMPLETE
832,832,0.188067,2026-03-07 05:19:56.257032,2026-03-07 05:20:32.375884,0 days 00:00:36.118852,0.353506,0.504133,seeds,97,118.838785,1.598024,37.485915,0.420642,COMPLETE
704,704,0.188109,2026-03-07 05:15:21.517040,2026-03-07 05:15:44.865020,0 days 00:00:23.347980,0.354766,0.548414,seeds,68,16.805789,5.398335,37.906043,0.445922,COMPLETE
799,799,0.188268,2026-03-07 05:17:34.439975,2026-03-07 05:18:11.975474,0 days 00:00:37.535499,0.374607,0.519837,seeds,91,20.994222,3.277896,52.588546,0.409538,COMPLETE
361,361,0.188287,2026-03-07 04:59:18.389655,2026-03-07 05:00:17.856013,0 days 00:00:59.466358,0.308716,0.500086,seeds,90,217.713533,11.935586,36.741048,0.435749,COMPLETE
550,550,0.188340,2026-03-07 05:08:33.017084,2026-03-07 05:09:01.460416,0 days 00:00:28.443332,0.375717,0.485768,seeds,72,211.302444,7.302716,33.351364,0.440366,COMPLETE
675,675,0.188363,2026-03-07 05:14:27.118707,2026-03-07 05:14:57.411538,0 days 00:00:30.292831,0.379720,0.460420,seeds,98,16.621569,1.747527,38.168762,0.403896,COMPLETE
879,879,0.188365,2026-03-07 05:21:58.760370,2026-03-07 05:22:20.385901,0 days 00:00:21.625531,0.361203,0.547310,seeds,114,17.684617,5.505772,37.355211,0.444824,COMPLETE
204,204,0.188371,2026-03-07 04:52:59.533648,2026-03-07 04:53:20.247208,0 days 00:00:20.713560,0.320349,0.498580,seeds,47,154.963562,8.542859,8.548269,0.459351,COMPLETE
618,618,0.188373,2026-03-07 05:10:45.578551,2026-03-07 05:11:09.542916,0 days 00:00:23.964365,0.387443,0.507357,seeds,88,13.490536,8.873495,31.159116,0.444990,COMPLETE


## LogisticRegression

In [9]:
def objective(trial, data=df_train, target=df_train["target"]):
    param = {
        'C': trial.suggest_float('reg_lambda', 1e-3, 100.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 100.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1),
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.3, 1),
        'subsample': trial.suggest_float('subsample', 0.4, 1),
        'min_child_weight': trial.suggest_float('min_child_weight', 1e-3 , 300),
        'feats': trial.suggest_categorical("feats", list(feats_dict.keys())),
    }
    
        
    train = data.copy()

    FEATURES = feats_dict[param["feats"]]


    num_pipe = Pipeline([('sel', tml.DtypeSel('numeric')),
                        ('imputer', tml.DfImputer(strategy='mean')),
                        ])

    processing_pipe = tml.FeatureUnionDf(transformer_list=[('num', num_pipe),
                                                        #('cat_means', cat_pipe)
                                                        ])
        
    model = LogisticRegression(C=param["C"], random_state=34, max_iter=10000, n_jobs=-1)

    pipe = Pipeline([ #("fe", FeatEng(formula=param["formula"])),
                        ("processing", processing_pipe),
                        ("scaler", tml.DfScaler()),

                    ("model", model)])
    

    cvscore = tml.CrossValidate(data=train[FEATURES], target=target, cv=kfolds, estimator=pipe, predict_proba=True)
    oof, _ = cvscore.score()

    score = brier_score_loss(target, y_prob=oof)
    
    return score

In [11]:
sampler = TPESampler(seed=645)  # Make the sampler behave in a deterministic way.

study = optuna.create_study(direction='minimize', sampler=sampler)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=2000, n_jobs=-1)
print('Number of finished trials:', len(study.trials))
print('Best trial:', study.best_trial.params)

study.trials_dataframe().sort_values('value', ascending=True).head(20)

Number of finished trials: 2000
Best trial: {'reg_lambda': 0.7208134861727786, 'reg_alpha': 28.92221114959707, 'colsample_bytree': 0.6575402094206728, 'colsample_bylevel': 0.44237325402371264, 'subsample': 0.4492681577515264, 'min_child_weight': 42.496464425237924, 'feats': 'all_delta'}


,number,value,datetime_start,datetime_complete,duration,params_colsample_bylevel,params_colsample_bytree,params_feats,params_min_child_weight,params_reg_alpha,params_reg_lambda,params_subsample,state
122,122,0.182708,2026-03-08 16:43:09.798511,2026-03-08 16:43:24.296954,0 days 00:00:14.498443,0.442373,0.657540,all_delta,42.496464,28.922211,0.720813,0.449268,COMPLETE
44,44,0.182710,2026-03-08 16:41:23.191715,2026-03-08 16:41:37.755465,0 days 00:00:14.563750,0.958925,0.784753,all_delta,38.869996,47.748636,0.733782,0.888776,COMPLETE
129,129,0.182712,2026-03-08 16:43:20.595071,2026-03-08 16:43:36.055486,0 days 00:00:15.460415,0.547751,0.719934,all_delta,15.412447,79.430333,0.873745,0.774534,COMPLETE
38,38,0.182712,2026-03-08 16:41:15.841957,2026-03-08 16:41:27.020469,0 days 00:00:11.178512,0.440152,0.704155,all_delta,31.827618,27.228396,0.927101,0.402283,COMPLETE
39,39,0.182716,2026-03-08 16:41:20.075206,2026-03-08 16:41:32.879492,0 days 00:00:12.804286,0.445057,0.705668,all_delta,35.252257,31.629022,0.999252,0.402913,COMPLETE
37,37,0.182718,2026-03-08 16:41:14.962172,2026-03-08 16:41:24.724301,0 days 00:00:09.762129,0.455368,0.701803,all_delta,23.422412,28.218358,0.992595,0.430088,COMPLETE
280,280,0.182720,2026-03-08 16:45:26.260966,2026-03-08 16:45:42.061405,0 days 00:00:15.800439,0.982108,0.704392,all_delta,104.658918,35.897774,1.045842,0.432168,COMPLETE
279,279,0.182725,2026-03-08 16:45:25.499761,2026-03-08 16:45:43.367824,0 days 00:00:17.868063,0.977181,0.688165,all_delta,9.947321,59.359773,1.110484,0.626554,COMPLETE
659,659,0.182725,2026-03-08 16:52:33.389152,2026-03-08 16:52:46.108740,0 days 00:00:12.719588,0.965323,0.776162,all_delta,36.116308,46.966008,1.091437,0.408851,COMPLETE
117,117,0.182731,2026-03-08 16:43:06.307920,2026-03-08 16:43:21.462433,0 days 00:00:15.154513,0.601846,0.648504,all_delta,38.849695,28.309613,1.195548,0.450419,COMPLETE
